
# Agentic AI on Gemini Enterprise Agent Platform
### GCP Training Program — Day 4, Module 7: Building, Deploying & Governing an Agent

**Problem statement:** we're extending the "poor man's agent" from the companion Gen AI notebook
(Section 4.4's manual function-calling round trip) into an actual, deployed agent. Our agent is
deliberately trivial — a **currency exchange assistant** with one real tool (a live exchange-rate
lookup) — so the class's attention stays on the **platform mechanics** (how an agent is built,
deployed, remembered, governed, and evaluated) rather than on the agent's own logic.

**What this notebook covers**, mapped to Agent Platform's four pillars (**Build, Scale, Govern,
Optimize**):

| Pillar | Component | What we do |
|---|---|---|
| Build | **Agent Development Kit (ADK)** | Define the currency agent's instructions and tool, run it locally |
| Build | **MCP Servers** | Connect a real Model Context Protocol tool server instead of a hand-written function |
| Scale | **Agent Runtime** (formerly Agent Engine) | Deploy the agent to a managed, autoscaling runtime |
| Scale | **Sessions & Memory Bank** | Give the agent persistent memory across separate conversations |
| Govern | **Agent Registry** | See the agent auto-catalogued after deployment |
| Govern | **Agent Identity, Policies, Gateway, Security** | Guided console walkthrough of governance controls |
| Optimize | **Evaluation** | Score the agent's tool-calling accuracy against a small test set |
| Build | **Agent Garden** | Guided walkthrough of prebuilt agent templates |

## ⚠️ Terminology note
Everything in this notebook lives under **Agent Platform > Agents** in the console — a separate
top-level section from **Agent Platform > Models**, which the companion Gen AI notebook covers.
"Agent Engine" (the old Vertex AI name) is now called **Agent Runtime**.

## ⚠️ Cost & billing note
The **Agent Runtime deployment** (Section 3) is the one resource here that keeps running and
billing after you create it — unlike a training job, it doesn't stop on its own. **Delete it in
the Cleanup section (Section 9) when you're done with the demo**, not at the end of the day.

**This notebook is fully self-contained.** Authenticate → Configure → everything else runs against
your own project.


## 1. Authenticate & Install

In [ ]:

import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab.")
else:
    print("Not running in Colab — assuming gcloud Application Default Credentials are set up.")
    print("If this is your first time, run in a terminal:")
    print("  gcloud auth application-default login")


In [ ]:

# The [agent_engines,adk] extras pull in both the ADK framework and the Agent Runtime deployment
# helpers, all under the same google-cloud-aiplatform package used in the companion notebook.
!pip install --upgrade --quiet "google-cloud-aiplatform[agent_engines,adk]>=1.112"


In [ ]:

import IPython
IPython.Application.instance().kernel.do_shutdown(True)


Install & Import
`google-genai` is a single, small package — replaces the generative-AI portions of
`google-cloud-aiplatform` entirely. `google-cloud-aiplatform` itself is still needed for Section 6
(Vector Search), which is a separate, non-generative-AI module.

> **Note on a pip warning you may see:** pip may report a conflict involving `google-ai-generativelanguage`
> and `protobuf`. That package is a legacy dependency baked into Colab's base image (from the older
> `google-generativeai` SDK) that this notebook never imports — we use `google-genai` exclusively. The
> conflict is safe to ignore; the verification cell right after the install confirms it.

In [ ]:
# Verify the packages this notebook actually uses import cleanly, regardless of the
# google-ai-generativelanguage/protobuf warning above (that package is unused here).
from google import genai
from google.cloud import aiplatform
import numpy as np

print("google-genai imported OK:", genai.__name__)
print("google-cloud-aiplatform imported OK:", aiplatform.__name__)
print("numpy imported OK, version:", np.__version__)
print("\nAll packages this notebook uses are working — the earlier pip warning can be ignored.")

## 2. Configure Your Project & Region

In [ ]:

PROJECT_ID = input("Enter your GCP Project ID: ").strip()

REGION = input("Enter your Agent Platform region [default: us-central1]: ").strip()
if not REGION:
    REGION = "us-central1"

BUCKET_NAME = f"{PROJECT_ID}-agentic-demo-bucket"
STAGING_BUCKET = f"gs://{BUCKET_NAME}"

!gcloud config set project {PROJECT_ID}
!gcloud services enable aiplatform.googleapis.com storage.googleapis.com --project {PROJECT_ID}

import vertexai
client = vertexai.Client(project=PROJECT_ID, location=REGION)

print("Project:", PROJECT_ID)
print("Region:", REGION)
print("Staging bucket:", STAGING_BUCKET)


In [ ]:

from google.cloud import storage

storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.lookup_bucket(BUCKET_NAME)
if bucket is None:
    bucket = storage_client.create_bucket(BUCKET_NAME, location=REGION)
    print(f"Created bucket {STAGING_BUCKET}")
else:
    print(f"Using existing bucket {STAGING_BUCKET}")



> 🖥️ **Check it in the console:** **IAM & Admin > IAM** → confirm your account has the
> **Agent Platform User** role (or Owner/Editor) before continuing — the deployment step in
> Section 3 needs it.


In [ ]:
import os

# Tell ADK's underlying google-genai client to use the Vertex AI / Agent Platform backend
# (which uses our gcloud ADC credentials) instead of defaulting to the Gemini Developer API
# (which requires a separate API key we haven't set).
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = REGION

print("ADK configured to use Vertex AI / Agent Platform backend for:", PROJECT_ID, REGION)


## 3. Build: Define the Agent With ADK

**What this does:** `google.adk.agents.Agent` is the core ADK building block — a name, a model, an
instruction (system prompt), and a list of tools. This mirrors exactly the function-calling pattern
from the companion notebook's Section 4.4, except ADK handles the "call tool → feed result back →
repeat" loop for you automatically instead of you writing that loop by hand.

Our one tool, `get_exchange_rate`, calls a real public API (no API key needed) — chosen specifically
so the agent's tool call has a genuine, verifiable external effect the class can see working, not a
canned/fake response.


In [ ]:

def get_exchange_rate(currency_from: str = "USD", currency_to: str = "EUR", currency_date: str = "latest"):
    '''Retrieves the exchange rate between two currencies on a specified date.

    Args:
        currency_from: The base currency code, e.g. "USD".
        currency_to: The target currency code, e.g. "EUR".
        currency_date: The date to get rates for, in YYYY-MM-DD format, or "latest".

    Returns:
        A dict with the exchange rate data.
    '''
    import requests
    response = requests.get(
        f"https://api.frankfurter.app/{currency_date}",
        params={"from": currency_from, "to": currency_to},
    )
    return response.json()

# Quick local sanity check before wrapping it in an agent at all
print(get_exchange_rate("USD", "INR"))


In [ ]:

from google.adk.agents import Agent

root_agent = Agent(
    model="gemini-2.5-flash",
    name="currency_exchange_agent",
    instruction=(
        "You are a currency exchange assistant. When asked about exchange rates, "
        "use the get_exchange_rate tool rather than answering from memory, since rates change daily. "
        "Give a short, direct answer citing the specific rate returned by the tool."
    ),
    tools=[get_exchange_rate],
)

print("Agent defined:", root_agent.name)



### 3.1 Run the Agent Locally First
**Why locally first:** exactly like testing a training script on a small sample before submitting
a full Custom Training job, testing the agent's reasoning and tool-calling loop locally — before
deploying it anywhere — catches obvious mistakes far faster than debugging through a deployed
endpoint's logs.


In [ ]:

from google.adk.runners import InMemoryRunner
from google.genai import types as genai_types

runner = InMemoryRunner(agent=root_agent)
session = runner.session_service.create_session_sync(
    app_name=runner.app_name, user_id="local-test-user"
)

user_message = genai_types.Content(
    role="user", parts=[genai_types.Part(text="What is the exchange rate from US dollars to SEK today?")]
)

for event in runner.run(user_id="local-test-user", session_id=session.id, new_message=user_message):
    if event.content and event.content.parts:
        for part in event.content.parts:
            if part.text:
                print("Agent:", part.text)
            if part.function_call:
                print(f"[tool call] {part.function_call.name}({dict(part.function_call.args)})")
            if part.function_response:
                print(f"[tool result] {part.function_response.response}")



> 🖥️ **Check it in the console:** nothing to check yet — this ran entirely in your Colab session,
> which is the point of testing locally before touching any cloud resource.


## 4. Build: Connect an MCP Tool Server

**What MCP is:** the Model Context Protocol is an open standard for exposing tools to an agent over
a consistent interface — instead of writing a Python function per tool (as we did in Section 3),
you point the agent at an **MCP server** that already exposes a whole set of related tools.

**What we'll do:** connect to **DeepWiki MCP** (`mcp.deepwiki.com`), a free, public, no-auth-required
server that exposes AI-powered documentation lookup for any public GitHub repository. This uses the
**Streamable HTTP** transport (a normal HTTP connection, not a spawned subprocess) — which also
sidesteps a Colab-specific limitation with stdio-based MCP servers spawning local subprocesses.

Agent Platform also maintains a hosted catalog of prebuilt MCP servers (BigQuery, Google Maps, and
others) under **Agent Platform > Agents > Build > MCP Servers**, connected the same way, just
pointing at Google's endpoint instead of DeepWiki's.

In [ ]:
!pip install --quiet mcp

In [ ]:
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams

print("Connecting to DeepWiki MCP server...")
deepwiki_mcp = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url="https://mcp.deepwiki.com/mcp",
    ),
)

discovered_tools = await deepwiki_mcp.get_tools()

if not discovered_tools:
    print("⚠️ No tools discovered — the connection likely failed silently. "
          "Try re-running this cell once; if it's still empty, mcp.deepwiki.com may be down.")
else:
    print(f"✅ Discovered {len(discovered_tools)} tools from the MCP server:")
    for t in discovered_tools:
        print(f"  - {t.name}: {t.description}")

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.genai import types as genai_types

docs_agent = Agent(
    model="gemini-2.5-flash",
    name="repo_docs_mcp_agent",
    instruction=(
        "You answer questions about GitHub repositories using the DeepWiki tools available to you. "
        "Use ask_question for the repo the user names, rather than answering from memory."
    ),
    tools=[deepwiki_mcp],
)

runner = InMemoryRunner(agent=docs_agent)
session = runner.session_service.create_session_sync(app_name=runner.app_name, user_id="mcp-test-user")

user_message = genai_types.Content(
    role="user",
    parts=[genai_types.Part(text="In the google/adk-python repo, what transport protocols does MCPToolset support?")],
)

print("Sending query to agent...\n")
event_count = 0

for event in runner.run(user_id="mcp-test-user", session_id=session.id, new_message=user_message):
    event_count += 1
    print(f"--- Event {event_count} (author: {event.author}) ---")

    function_calls = event.get_function_calls()
    if function_calls:
        for fc in function_calls:
            print(f"  [MCP TOOL CALL] {fc.name}({dict(fc.args)})")

    function_responses = event.get_function_responses()
    if function_responses:
        for fr in function_responses:
            print(f"  [MCP TOOL RESULT] {str(fr.response)[:500]}")

    if event.content and event.content.parts:
        for part in event.content.parts:
            if part.text:
                print(f"  [TEXT] {part.text}")

    if not function_calls and not function_responses and not (event.content and event.content.parts):
        print(f"  [RAW EVENT, no recognized content] {event}")

print(f"\nTotal events received: {event_count}")
if event_count == 0:
    print("⚠️ Zero events received — the runner call itself may have failed silently. "
          "Check that `session` was created successfully in the cell above.")

**What to look for:** the tool call above should show the agent invoking `ask_question` (or
`read_wiki_contents`) with the repo name and your question as arguments — real MCP protocol traffic
against a live public server, not a canned response. The answer should reflect actual content from
that repository's indexed documentation.

> 🖥️ **Check it in the console:** **Agent Platform > Agents > Build > MCP Servers** → browse the
> hosted catalog (BigQuery, Google Maps, etc.) — the same `McpToolset` + `StreamableHTTPConnectionParams`
> pattern above applies to any of these, just swapping the `url` for Google's hosted endpoint.


## 5. Scale: Deploy the Agent to Agent Runtime

**What this does:** `client.agent_engines.create()` packages the agent (and its dependencies) and
deploys it to **Agent Runtime** — a managed, autoscaling execution environment, distinct from your
Colab kernel. This is the same conceptual step as Custom Training's `CustomJob.run()` in the
companion churn notebook: your code runs on Google's managed infrastructure, not locally.

Deploying an ADK agent to Agent Platform automatically generates a managed **session** resource and
registers the agent in the **Agent Registry** (Section 7) — both happen as a side effect of this
one step, with no separate action needed.


In [ ]:

from vertexai import agent_engines, types as vertexai_types

app = agent_engines.AdkApp(agent=root_agent)

remote_agent = client.agent_engines.create(
    agent=app,
    config={
        "requirements": ["google-cloud-aiplatform[agent_engines,adk]"],
        "staging_bucket": STAGING_BUCKET,
        "display_name": "currency-exchange-agent",
        "identity_type": vertexai_types.IdentityType.AGENT_IDENTITY,
    },
)

print("Deployed agent resource name:", remote_agent.api_resource.name)



This step typically takes **several minutes** — Agent Runtime is provisioning a dedicated managed
environment, similar in spirit to the VM-provisioning wait we saw for Custom Training jobs, though
Agent Runtime is specifically engineered for much faster (often sub-second) cold starts on
subsequent invocations once deployed.

> 🖥️ **Check it in the console:** **Agent Platform > Agents > Scale > Deployments** → your
> `currency-exchange-agent` should appear here, with its status, region, and identity details.



### 5.1 Query the Deployed Agent
**What this does:** same conversational request as the local test in Section 3.1, but now routed
through the deployed Agent Runtime endpoint instead of running in this notebook's own process —
proving the deployment actually works end-to-end.


In [ ]:

import asyncio

async def query_remote_agent(message, user_id="remote-test-user"):
    async for event in remote_agent.async_stream_query(user_id=user_id, message=message):
        print(event)

await query_remote_agent("What is the exchange rate from US dollars to SEK today?")



## 6. Scale: Sessions & Memory Bank

**Sessions vs. Memory Bank — the distinction that matters:**
- A **Session** holds the back-and-forth of *one* ongoing conversation — the same role
  `client.chats.create()`'s history played in the companion notebook's Section 4.2, just now
  managed by Agent Runtime instead of the Gen AI SDK client.
- **Memory Bank** is different: it *extracts and stores* durable facts from past sessions (e.g. "this
  user usually asks about USD-to-INR rates") so a **brand-new session**, days later, can still
  recall them — persistent memory *across* sessions, not just within one.

We demonstrate this concretely: tell the agent something in Session A, start a completely separate
Session B, and check whether it remembers.


In [ ]:
user_id = "memory-demo-user"
agent_engine_name = remote_agent.api_resource.name

# Session A: mention a standing preference
session_a = client.agent_engines.sessions.create(name=agent_engine_name, user_id=user_id)
async for event in remote_agent.async_stream_query(
    user_id=user_id,
    session_id=session_a.response.name.split("/")[-1],
    message="For all my future questions, I'm always converting from British Pounds (GBP), not USD.",
):
    print(event)

In [ ]:
# Instead of hoping automatic background extraction has already run by the time we check,
# explicitly trigger it and wait for completion — this is the "improvement" step: turning an
# implicit, timing-dependent behavior into a concrete, verifiable one.
generate_result = client.agent_engines.memories.generate(
    name=agent_engine_name,
    vertex_session_source={"session": session_a.response.name},
    config={"wait_for_completion": True},
)
print("Memory generation triggered and completed.")
print(generate_result)

In [ ]:
results = client.agent_engines.memories.retrieve(
    name=agent_engine_name,
    scope={"user_id": user_id},
)

memories_list = list(results)
print(f"Found {len(memories_list)} memory/memories for user '{user_id}':\n")
for r in memories_list:
    print(f"  - {r.memory.fact}")

if not memories_list:
    print("  (none — Memory Bank likely didn't extract anything memorable from that one exchange;")
    print("   see the note below on why that can legitimately happen.)")

**Why Cell 3 might legitimately show zero memories:** Memory Bank only extracts what it judges to
be a durable, reusable fact about the user — matching topics configured for your Memory Bank
instance — not a transcript of everything said. A single "always use GBP" statement is a good
candidate, but this is a judgment call made by Gemini under the hood, not a guaranteed rule. If
Cell 3 shows nothing, that's Memory Bank correctly declining to store something it doesn't
consider durable — not a bug. Compare this with `direct_contents_source` (documented under
Memory Bank's Generate Memories page), which lets you *directly* upload pre-extracted facts
instead of relying on automatic extraction — the reliable path for anything your application
knows for certain should be remembered, rather than leaving it to inference.

In [ ]:
session_b = client.agent_engines.sessions.create(name=agent_engine_name, user_id=user_id)
async for event in remote_agent.async_stream_query(
    user_id=user_id,
    session_id=session_b.response.name.split("/")[-1],
    message="What's today's exchange rate to Japanese Yen?",
):
    print(event)

print("\nBecause we explicitly generated and verified the memory above (Cells 2-3), this is now a")
print("clean test of recall, not a guess about background-extraction timing.")


**Note on timing:** Memory Bank extraction from a session isn't always instantaneous — it may run
as an asynchronous step after the session ends, so an immediate check like the one above can
sometimes predate the memory actually being written. If Session B doesn't show the expected recall,
this is the most likely reason, not a code error.

> 🖥️ **Check it in the console:** **Agent Platform > Agents > Scale > Sessions** → both
> `session_a` and `session_b` should be listed under `memory-demo-user`. **Agent Platform > Agents >
> Scale > Memory Bank** → look for a memory entry referencing the GBP preference, generated from
> Session A.



## 7. Govern: Agent Registry

**What this is:** a centralized catalog for discovering, tracking, and managing every agent, tool,
and MCP server across your organization — the agent equivalent of the Model Registry from the
companion churn-prediction notebook. Deploying an agent to Agent Runtime (Section 5) automatically
registers it here; there's no separate registration step.


In [ ]:

agents_list = client.agent_engines.list()
for a in agents_list:
    print(f"{a.api_resource.display_name}  ->  {a.api_resource.name}")



> 🖥️ **Check it in the console:** **Agent Platform > Agents > Govern > Agent Registry** →
> `currency-exchange-agent` should be listed here, matching the output above, with its version
> history and the tools/MCP servers it depends on.



## 8. Govern: Agent Identity, Policies, Gateway & Security (Guided Walkthrough)

**Why this is a walkthrough, not a live exercise:** these are organization-level governance controls
— meaningful in a multi-agent, multi-team environment, not something a single toy agent in a
training notebook can usefully demonstrate live. We already benefited from one of them without
seeing it: setting `identity_type=AGENT_IDENTITY` in Section 5 gave our deployed agent a unique,
auditable cryptographic identity, rather than running under a generic shared service account.

- **Agent Identity** — every deployed agent gets its own verifiable identity (what we set explicitly
  in Section 5), enabling fine-grained IAM permissions and a clear audit trail per agent, the same
  way each service account in a project has its own permissions and logs.
- **Policies** — organization-wide rules like **Content Protection** (blocking sensitive data from
  leaving an agent's responses) and **Semantic Governance** (constraining what topics/actions an
  agent is allowed to engage with), applied centrally rather than coded into every agent individually.
- **Agent Gateway** — a central enforcement point all agent tool calls pass through, handling
  authentication and applying Model Armor protections against prompt injection and data leakage —
  conceptually similar to an API gateway in front of a set of microservices, but purpose-built for
  agent-to-tool traffic.
- **Security** — combines Agent Identity, Model Armor, and continuous threat detection (anomaly
  detection using statistical models plus an LLM-as-judge, and explicit threat detection for things
  like reverse-shell attempts) into one dashboard, backed by Security Command Center.

> 🖥️ **Check it in the console:**
> - **Agent Platform > Agents > Govern > Agent Registry** → click into `currency-exchange-agent` →
>   its **Identity** tab shows the cryptographic identity created in Section 5.
> - **Agent Platform > Agents > Govern > Policies** → browse available policy templates (Content
>   Protection, Semantic Governance) even without applying one to this demo agent.
> - **Agent Platform > Agents > Govern > Gateway** → see how gateway instances route and log tool
>   calls across all registered agents.
> - **Agent Platform > Agents > Govern > Security** → the dashboard that would show anomaly/threat
>   detection findings once an agent has meaningful production traffic.



## 9. Optimize: Evaluate the Agent

**What this does:** rather than eyeballing a handful of manual test messages (as we did in Sections
3.1 and 5.1), we run a small, structured test set through the agent and score whether it actually
called the right tool with the right arguments — the same spirit as the classification metrics
(`accuracy`, `f1_score`) logged as Experiment runs in the companion churn notebook, just measuring
tool-calling correctness instead of prediction correctness.


In [ ]:
from google import genai

genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

In [ ]:
eval_cases = [
    {"query": "What's the USD to EUR rate?", "expected_from": "USD", "expected_to": "EUR"},
    {"query": "Convert Japanese Yen to Australian Dollars.", "expected_from": "JPY", "expected_to": "AUD"},
    {"query": "How many Canadian Dollars is one US Dollar worth?", "expected_from": "USD", "expected_to": "CAD"},
]

correct = 0
for case in eval_cases:
    response = genai_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=case["query"],
        config={"tools": [{"function_declarations": [{
            "name": "get_exchange_rate",
            "description": "Get exchange rate between two currencies.",
            "parameters": {
                "type": "object",
                "properties": {
                    "currency_from": {"type": "string"},
                    "currency_to": {"type": "string"},
                },
            },
        }]}]},
    )
    call = response.candidates[0].content.parts[0].function_call
    args = dict(call.args) if call else {}
    is_correct = (
        args.get("currency_from") == case["expected_from"]
        and args.get("currency_to") == case["expected_to"]
    )
    correct += is_correct
    print(f"Query: {case['query']}")
    print(f"  Expected: {case['expected_from']} -> {case['expected_to']}, Got: {args}")
    print(f"  {'✅ Correct' if is_correct else '❌ Incorrect'}\n")

print(f"Tool-calling accuracy: {correct}/{len(eval_cases)} ({correct/len(eval_cases):.0%})")


**Scaling this up:** three hand-written test cases is enough to demonstrate the pattern; a real
evaluation suite under **Agent Platform > Agents > Optimize > Evaluation** runs this same idea
against much larger test sets, can compare multiple agent versions side by side, and supports
running against *live* production traffic samples rather than only a static test set — the agent
equivalent of comparing multiple model versions in the companion notebook's Experiments section.

> 🖥️ **Check it in the console:** **Agent Platform > Agents > Optimize > Evaluation** → the managed
> evaluation UI runs the same kind of check at scale, with built-in support for larger test sets and
> side-by-side comparison across agent versions.



## 10. Agent Garden (Guided Walkthrough)

**What this is:** a library of prebuilt agent templates — the agent equivalent of Model Garden from
the companion notebook, but for entire agents rather than individual models. Templates cover common
enterprise patterns: financial analysis, invoice processing, code modernization, and more.

**Why this is a walkthrough:** browsing and adapting a template is a console-driven exploration
step, not something meaningfully demonstrated on this notebook's toy currency-exchange example. If
you wanted to build something closer to a real use case after this training, Agent Garden is the
recommended starting point rather than building an agent from a blank ADK project.

> 🖥️ **Check it in the console:** **Agent Platform > Agents > Build > Agent Garden** → browse the
> available templates and note which ones are closest to a real problem in your own organization —
> a good exercise for a follow-up session beyond this notebook.



## 11. Cleanup

The deployed Agent Runtime resource (Section 5) is the one thing in this notebook that keeps
running and billing — delete it now if you're done with the demo. This is guarded by a
`CONFIRM_DELETE` flag, same pattern as the companion notebook's cleanup section, so it can't run by
accident.


In [ ]:

CONFIRM_DELETE = True  # <-- set to True to actually delete resources

if not CONFIRM_DELETE:
    print("CONFIRM_DELETE is False — nothing will be deleted. "
          "Set CONFIRM_DELETE = True above and re-run this cell when you're ready.")


In [ ]:

if CONFIRM_DELETE:
    # Delete the deployed agent from Agent Runtime (this also removes it from the Agent Registry)
    try:
        remote_agent.delete()
        print("Deleted deployed agent from Agent Runtime.")
    except Exception as e:
        print("Agent deletion skipped/failed:", e)


In [ ]:

if CONFIRM_DELETE:
    # Delete the staging bucket used for deployment artifacts
    try:
        bucket.delete(force=True)
        print(f"Deleted bucket {STAGING_BUCKET}.")
    except Exception as e:
        print("Bucket cleanup skipped/failed:", e)

    print("\nCleanup complete. Sessions and Memory Bank entries tied to the deleted agent are")
    print("removed along with it. The Agent Registry listing should no longer show this agent.")


In [ ]:
if CONFIRM_DELETE:
    # Catch-all: delete every Agent Runtime deployment matching this notebook's display name,
    # rather than relying on the `remote_agent` variable, which may not reflect every deployment
    # created across multiple runs of Section 5 (e.g. after adding the search_memories tool).
    deployed_agents = client.agent_engines.list()
    matches = [a for a in deployed_agents if a.api_resource.display_name == "currency-exchange-agent"]

    if not matches:
        print("No deployed agents found matching 'currency-exchange-agent'.")
    for a in matches:
        try:
            a.delete()
            print(f"Deleted: {a.api_resource.name}")
        except Exception as e:
            print(f"Cleanup skipped/failed for {a.api_resource.name}: {e}")


> 🖥️ **Final console check:** **Agent Platform > Agents > Scale > Deployments** and
> **Agent Platform > Agents > Govern > Agent Registry** should both show `currency-exchange-agent`
> gone. If either still lists it, delete it manually from the console rather than re-running this
> notebook's cells against a resource that's already partially cleaned up.
